<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-09-multimodal-and-pretrained/lesson-9.4-media-studio/notebooks/GCP_Capstone_9.4_MediaStudio.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.4 The DocuMind Media Studio — Generate, Describe, Segment, Upload: the Whole Pipe, on the Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

The Studio end to end. The buckets as Terraform made them; an image generated through the API's route with its audit row and usage row read back; caption, alt text and the table as JSON on Figure 3, beside the caption the worker wrote; the town hall segmented with the worker's own prompt and schema, beside the segments the corpus holds; the MM:SS coverage check on fixtures; diarisation against the committed transcript; the 32 MB wall; and the signed PUT that is now an ingest - a probe image uploaded straight to GCS and polled until the lane cites it. The server half is the kit's `media.py`, read from the clone.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 Pillow==12.3.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media, the routes, the record


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


In [ ]:
import datetime

def audit_rows(action: str, tenant: str = TENANT, minutes: int = 30) -> list[dict]:
    """The audit trail, read back. shared/audit_log.py writes ONE JSON object per event into the
    retention-locked audit bucket, named {year}/{month}/{day}/{tenant}/{action}-{id}.json - so the
    record of who generated what is a listing, not a query. The media bucket is only a cache; this
    is the record."""
    now = datetime.datetime.now(datetime.timezone.utc)
    since = now - datetime.timedelta(minutes=minutes)
    rows = []
    for day in sorted({since.strftime("%Y/%m/%d"), now.strftime("%Y/%m/%d")}):
        for blob in gcs.list_blobs(f"{PROJECT_ID}-audit", prefix=f"{day}/{tenant}/{action}-"):
            if blob.updated and blob.updated < since:
                continue
            rows.append(json.loads(blob.download_as_text()))
    return sorted(rows, key=lambda r: r.get("ts", ""))


## Cell 2: The buckets, as deployed
Read, never created. The media bucket is a 30-day cache; the uploads bucket is the corpus, with the notification.


In [ ]:
# THE BUCKET, AS DEPLOYED. storage.tf made it: India region, uniform access, a 30-day delete rule
# (generated media is a cache; the audit row is the record) and a CORS block for browser PUTs. Nothing
# here creates or changes it - read it, and read the uploads bucket beside it, because that is where a
# signed upload now lands (Cell 9): the bucket eventarc.tf watches.
for name in (MEDIA_BUCKET, UPLOAD_BUCKET):
    b = gcs.get_bucket(name)
    rule = next(iter(b.lifecycle_rules), {})
    print(f"{name}: {b.location} | uniform access: {b.iam_configuration.uniform_bucket_level_access_enabled} | "
          f"lifecycle: {rule.get('action', {}).get('type', '-')} after {rule.get('condition', {}).get('age', '-')} days | "
          f"CORS methods: {[c.get('method') for c in b.cors] or '-'}")
media = gcs.get_bucket(MEDIA_BUCKET)
assert next(iter(media.lifecycle_rules), {}).get("condition", {}).get("age") == 30, "the media bucket is a 30-day cache by design"


## Cell 3: Generate through the route, and read the record and the meter back


In [ ]:
import hashlib
from io import BytesIO
from PIL import Image
from IPython.display import display

# GENERATE THROUGH THE ROUTE - the Studio's server half (services/rag-api/media.py), the one the UI's
# Studio tab calls. Roster first, one spend, cached under DEMO_MODE, an audit row and a usage row.
PROMPT = ("A clean diagram of DocuMind's ingestion pipeline: a document goes through OCR, chunking, embedding, "
          "then a vector index. Label each stage. Teal palette, white background, no small print.")
PROMPT_SHA = hashlib.sha256(PROMPT.encode()).hexdigest()[:32]
status, out = api("/v1/media/generate", {"prompt": PROMPT, "tenant_id": TENANT})
assert status == 200, (status, out)
print("blob:", out["blob"], "| cached:", out["cached"])
diagram_png = gcs_bytes(f"gs://{out.get('bucket', MEDIA_BUCKET)}/{out['blob']}")
display(Image.open(BytesIO(diagram_png)))

# The record and the meter, read back. The audit row: who, which tenant, the prompt's hash. The usage
# row goes to Cloud Logging as JSON (event=media, cost_usd) - tenant_daily's SQL reads it on the full
# profile, and `gcloud logging read` on the lean one.
for _ in range(6):
    rows = [r for r in audit_rows("media.generate") if r.get("meta", {}).get("prompt_sha") == PROMPT_SHA]
    if rows:
        break
    time.sleep(5)
assert rows, "no audit row for this prompt yet - re-run in a minute"
print("audit :", rows[-1]["actor"], rows[-1]["meta"])
since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=10)).strftime("%Y-%m-%dT%H:%M:%SZ")
r = subprocess.run(["gcloud", "logging", "read",
                    f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" AND jsonPayload.event="media" AND timestamp>="{since}"',
                    "--project", PROJECT_ID, "--limit", "5",
                    "--format=value(timestamp.date('%H:%M:%S'),jsonPayload.tenant,jsonPayload.user,jsonPayload.cost_usd,jsonPayload.cached)"],
                   capture_output=True, text=True)
print("usage :", r.stdout.strip() or "(logs lag a minute - run this cell again)")


## Cell 4: Caption, alt text and the table as JSON
A structured description on Figure 3 - and the worker's own caption of the same figure, which is the one the retriever matches.


In [ ]:
from pydantic import BaseModel

# CAPTION, ALT TEXT AND THE TABLE, AS JSON - the structured description a document index needs. Reading
# an image is a text task: gemini-3.6-flash, not the image model. On Figure 3, the table comes back as
# Markdown; and the worker wrote its own version of this at ingest (12.5), which is the one retrieve()
# matches against - compare them.
class Description(BaseModel):
    caption: str
    alt_text: str          # under 125 characters; never begins "image of"
    has_table: bool
    table_md: str          # GitHub-flavoured Markdown, "" when has_table is false
    entities: list[str]

d = gen.models.generate_content(model="gemini-3.6-flash",
    contents=[types.Part.from_uri(file_uri=FIG3, mime_type="image/png"),
              "Describe this image for a document-intelligence index. alt_text must be under 125 characters and must not "
              "begin 'image of'. If the image contains a table or a chart with values, transcribe the values as a "
              "GitHub-flavoured Markdown table in table_md; otherwise return ''."],
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Description)).parsed
print("caption :", d.caption)
print("alt     :", d.alt_text, f"({len(d.alt_text)} chars)")
print("table   :\n" + d.table_md)
assert d.has_table and "EMEA" in d.table_md and len(d.alt_text) < 125 and not d.alt_text.lower().startswith("image of")

hits = documind_tools.retrieve("Figure 3, revenue by region FY2025 versus FY2026", tenant_id=TENANT, top_k=8, brain="direct")
fig = next((c for c in hits.get("citations", []) if c.get("kind") == "figure" and c["source_uri"].endswith("fig3.png")), None)
assert fig, "no figure citation - is the media ingested? (make media, make ingest-corpus)"
print("\nthe worker's caption, which is what the retriever matched:", fig["quote"][:240], "...")


## Cell 5: The video, segmented the worker's way
The worker's prompt and schema, so this cell's segments and the corpus's are the same shape. Gated on the video being in the corpus.


In [ ]:
# THE VIDEO, SEGMENTED THE WORKER'S WAY. Point at it in GCS - the bytes never travel. The prompt and the
# Segment schema below are the ingest worker's own (services/ingest/main.py, _describe_media), so the
# segments this cell makes and the segments the corpus holds are the same shape, and you can put them
# side by side. media_resolution LOW: "what was said and roughly when" does not need to read slides.
class Segment(BaseModel):
    start: float
    end: float
    summary: str

def mmss(s: float) -> str:
    return f"{int(s) // 60:02d}:{int(s) % 60:02d}"

if HAS_VIDEO:
    r = gen.models.generate_content(model="gemini-3.6-flash",
        contents=[types.Part.from_uri(file_uri=VIDEO, mime_type="video/mp4"),
                  "Split this video into segments of at most 60 seconds. For each, give start and end in seconds and a "
                  "two-sentence summary of what is said and shown, quoting every number, percentage, amount and name "
                  "that is spoken exactly as it is said."],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Segment],
                                           media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW,
                                           thinking_config=types.ThinkingConfig(thinking_level="LOW")))
    mine = r.parsed or []
    print(f"this cell: {len(mine)} segments")
    for s in mine:
        print(f"  {mmss(s.start)}-{mmss(s.end)}  {s.summary[:90]}")
    hits = documind_tools.retrieve("What did the CFO say about EMEA revenue in the town hall?", tenant_id=TENANT, top_k=8, brain="direct")
    theirs = [c for c in hits.get("citations", []) if c.get("kind") == "segment"]
    print(f"\nthe corpus: {len(theirs)} segment citations for the EMEA question")
    for c in theirs[:3]:
        print(f"  {mmss(c['start'])}-{mmss(c['end'])}  {c['quote'][:90]}  <- media_url {c['media_url'].rsplit('/', 1)[-1]}")
    assert mine and theirs, "segments on one side only: the worker's are in the index once make ingest-corpus has run"
    assert any("EMEA" in c["quote"] for c in theirs), "no indexed segment mentions EMEA"
else:
    print("no townhall_2026_q1.mp4 in the corpus: make media MEDIA_ARGS=--video (Cloud Shell: Text-to-Speech + ffmpeg), then make ingest-corpus")


## Cell 6: The coverage check, on fixtures
A segment list that quietly skips forty seconds looks well-formed. This is the check that turns *the JSON parsed* into *the transcript is complete* - exercised in all three directions before it is trusted.


In [ ]:
# THE COVERAGE CHECK, exercised on fixtures - because a check you have never seen fail is a check you do
# not know works. The model is asked for no gaps and no overlaps; verify it, do not hope.
def mmss_to_s(s: str) -> int:
    """'03:20' -> 200. Raises on anything that is not MM:SS."""
    m, sec = s.split(':')
    return int(m) * 60 + int(sec)


def check_segments(segs: list[dict], runtime_s: int, tol: int = 2) -> list[str]:
    """The model is asked for no gaps and no overlaps. Verify it, do not hope.

    A segment list that quietly skips 40 seconds looks perfectly well-formed and
    loses whatever was said in those 40 seconds. This is the check that turns
    'the JSON parsed' into 'the transcript is complete'.
    """
    problems = []
    if not segs:
        return ['no segments returned']
    if mmss_to_s(segs[0]['start']) > tol:
        problems.append(f"starts at {segs[0]['start']}, not 00:00")
    for a, b in zip(segs, segs[1:]):
        gap = mmss_to_s(b['start']) - mmss_to_s(a['end'])
        if gap > tol:
            problems.append(f"gap {a['end']} -> {b['start']} ({gap}s)")
        if gap < -tol:
            problems.append(f"overlap {a['end']} -> {b['start']} ({-gap}s)")
    short = runtime_s - mmss_to_s(segs[-1]['end'])
    if short > tol:
        problems.append(f"ends at {segs[-1]['end']}, {short}s before the end")
    return problems


In [ ]:
# The check, exercised on fixtures - because a check you have never seen
# fail is a check you do not know works.
GOOD = [{'start': '00:00', 'end': '01:00', 'summary': 'intro', 'topics': ['q1']},
        {'start': '01:00', 'end': '02:00', 'summary': 'numbers', 'topics': ['rev']},
        {'start': '02:00', 'end': '03:00', 'summary': 'close', 'topics': ['q2']}]
GAP = [GOOD[0], {'start': '01:40', 'end': '03:00', 'summary': 'rest', 'topics': []}]
SHORT = GOOD[:2]

for name, segs, runtime in (('complete', GOOD, 180),
                            ('40s gap', GAP, 180),
                            ('ends early', SHORT, 180)):
    problems = check_segments(segs, runtime)
    verdict = 'OK' if not problems else '; '.join(problems)
    print(f'{name:12} -> {verdict}')

assert not check_segments(GOOD, 180), 'a complete list must pass'
assert check_segments(GAP, 180), 'a 40s gap must be caught'
assert check_segments(SHORT, 180), 'a short ending must be caught'
print()
print('coverage check works in all three directions')


## Cell 7: Diarisation, against the script
The transcript the video was synthesised from is committed: five turns, two speakers. Count what the model finds.


In [ ]:
# DIARISATION - who said what - AGAINST A GROUND TRUTH. The transcript the video was synthesised from
# is committed (deploy/evals/corpus/acme/townhall_2026_q1.md): five turns, two speakers. Ask the model
# for speaker-labelled turns and count. gemini-3.6-flash with a schema does this on the global endpoint;
# a dedicated transcription model would be cheaper per minute - but the global endpoint has no data
# residency, which for recorded customer voices is a DPA sentence, and why 9.3 keeps Chirp 3 in a region.
import re
class Turn(BaseModel):
    speaker: str
    start: float
    text: str

script = open(f"{KIT}/deploy/evals/corpus/acme/townhall_2026_q1.md", encoding="utf-8").read()
truth = re.findall(r"^([A-Z][a-z]+) \((CEO|CFO)\): ", script, re.M)
print("ground truth:", len(truth), "turns,", sorted({s for s, _ in truth}))

if HAS_VIDEO:
    r = gen.models.generate_content(model="gemini-3.6-flash",
        contents=[types.Part.from_uri(file_uri=VIDEO, mime_type="video/mp4"),
                  "Transcribe this recording as speaker-labelled turns. Label speakers by the name they are addressed by if "
                  "one is said, otherwise Speaker A and Speaker B. A new turn starts when the speaker changes."],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Turn],
                                           media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW))
    turns = r.parsed or []
    print(f"model: {len(turns)} turns, {sorted({t.speaker for t in turns})}")
    for t in turns[:6]:
        print(f"  {mmss(t.start)} {t.speaker:9} {t.text[:80]}")
    assert 2 <= len({t.speaker for t in turns}) <= 3, "diarisation did not find two speakers"
    assert abs(len(turns) - len(truth)) <= 2, f"{len(turns)} turns against {len(truth)} in the script"
else:
    print("(video absent - the cell compares the model's turns with the script's when the corpus holds it)")


## Cell 8: The 32 MB wall
Cloud Run answers an HTTP/1 body over 32 MB with 413. The signed URL keeps the bytes out of the container entirely - and, since Module 9 joined the lane, out of the media bucket and into the uploads bucket, where a PUT is an ingest.


In [ ]:
# The 32 MB wall.
#
# Cloud Run caps an HTTP/1 request body at 32 MB and answers anything larger
# with 413 Request Entity Too Large. A 40 MB video posted to your service does
# not fail slowly or partially - it never arrives.
#
# HTTP/2 lifts the cap, but signed URLs are still the right answer: the bytes
# go from the browser straight to GCS and never touch your container, so you
# are not paying Cloud Run CPU-seconds to be a pipe.
from datetime import timedelta

CLOUD_RUN_BODY_LIMIT = 32 * 1024 * 1024        # 32 MB, HTTP/1


def upload_route(size_bytes: int) -> str:
    """Which path a file of this size must take."""
    return 'direct-post' if size_bytes < CLOUD_RUN_BODY_LIMIT else 'signed-url'


def make_upload_url(blob_name: str, content_type: str, minutes: int = 15) -> str:
    """A one-file, one-content-type, time-boxed PUT URL.

    Scope it tightly: a signed URL is a bearer credential. Anyone holding it can
    write that object until it expires, so bind the content type and keep the
    window short.
    """
    blob = gcs.bucket(UPLOAD_BUCKET).blob(blob_name)
    return blob.generate_signed_url(version='v4', method='PUT',
                                    expiration=timedelta(minutes=minutes),
                                    content_type=content_type)


# The client then does exactly one thing:
#   PUT <signed_url>  with  Content-Type: video/mp4  and the file as the body.
# Your service never sees the bytes - it only ever sees the object name.


In [ ]:
# The routing decision, exercised. 40 MB is the gate's number.
for mb in (1, 31, 32, 40, 512):
    size = mb * 1024 * 1024
    print(f'{mb:>4} MB -> {upload_route(size)}')

assert upload_route(31 * 1024 * 1024) == 'direct-post'
assert upload_route(40 * 1024 * 1024) == 'signed-url', '40 MB must not go via Cloud Run'
print()
print('40 MB routes to the signed URL, which is what makes the upload succeed')


## Cell 9: The signed PUT, end to end
The route signs a one-object PUT into the uploads bucket; the notification hands the object to the worker; the worker captions, scans and indexes it; the lane cites it.


In [ ]:
from PIL import ImageDraw

# THE SIGNED PUT, END TO END. /v1/media/upload-url signs a one-object, one-content-type, 15-minute PUT
# into the UPLOADS bucket under the tenant's prefix - the bucket eventarc.tf watches. So the PUT is an
# ingest: the worker describes the image, scans its pixels, indexes the caption. This cell PUTs a small
# PNG it draws, then polls retrieve() until the figure comes back cited. The route used to sign into
# the media bucket, which nothing watches: stored, billed for thirty days, never indexed.
probe = Image.new("RGB", (900, 300), "white")
ImageDraw.Draw(probe).text((30, 110), "DocuMind lesson 9.4 probe: the signed PUT is an ingest", fill="#0f1729")
buf = BytesIO(); probe.save(buf, "PNG"); png = buf.getvalue()
NAME = "lesson_9_4_probe.png"

status, door = api(f"/v1/media/upload-url?filename={NAME}&content_type=image/png&tenant_id={TENANT}")
assert status == 200, (status, door)
print("signed for:", door["bucket"], door["blob"], "|", door["note"][:70], "...")
put = requests.put(door["url"], data=png, headers={"Content-Type": "image/png"}, timeout=60)
assert put.status_code in (200, 201), (put.status_code, put.text[:200])
print("PUT:", put.status_code, f"({len(png) // 1024} KB straight to GCS; the API never saw the bytes)")

found = None
for i in range(18):                                   # up to three minutes: the worker's caption, DLP, embedding
    hits = documind_tools.retrieve("DocuMind lesson 9.4 probe image about the signed PUT", tenant_id=TENANT, top_k=8, brain="direct")
    found = next((c for c in hits.get("citations", []) if c["source_uri"].endswith(NAME)), None)
    if found:
        break
    time.sleep(10)
assert found, "the probe never came back cited - read documind-ingest's log (a DLP or caption failure lands in the DLQ)"
print(f"\nindexed after ~{(i + 1) * 10}s: kind={found['kind']} | caption: {found['quote'][:120]}")


## Cell 10: The audit registry
The kit's writer, imported from the clone: it refuses a name it does not know.


In [ ]:
from shared import audit_log

# EVERY GENERATION WRITES AN AUDIT ROW, AND THE WRITER REFUSES A NAME IT DOES NOT KNOW. shared/audit_log.py
# is the one audit writer; emit() asserts the action is registered. media.generate and media.transcribe
# joined its set when the Studio joined the kit - an audit log that quietly drops events it does not
# recognise is worse than no audit log. This is the kit's own module, imported from the clone.
print("registered:", sorted(a for a in audit_log.AUDIT_ACTIONS if a.startswith(("media.", "doc.", "dlp."))))
assert {"media.generate", "media.transcribe"} <= audit_log.AUDIT_ACTIONS
try:
    audit_log.emit("media.remix", actor={"tenant_id": TENANT}, target={}, meta={})
    raise SystemExit("an unregistered action was accepted")
except AssertionError as e:
    print("refused:", e)


## Cell 11: The server half, from the clone


In [ ]:
# THE SERVER HALF, FROM THE CLONE. media.py is hand-written in the kit (no notebook owns it), and this
# lesson's cells called it: generate (roster, cache, audit, usage) and upload-url (a bare filename, a
# known content type, the uploads bucket). Read the guards; they are the auth-wiring gate's cases.
src = open(f"{KIT}/deploy/services/rag-api/media.py", encoding="utf-8").read()
for needle in ("enforce_membership(user[\"email\"], body.tenant_id)", "DEMO_MODE and blob.exists()", "audit_log.emit(",
               "UPLOAD_BUCKET", "content_type not in UPLOAD_TYPES", "generate_signed_url("):
    print(f"{'ok ' if needle in src else 'MISSING'}  {needle}")
print()
print("\n".join(src.splitlines()[:26]))


## Cell 12: What it costs


In [ ]:
# WHAT THE STUDIO COSTS, before you turn it on for five tenants. Verified 2026-09-04: images per IMAGE,
# transcription per MINUTE; re-verify on the pricing page - these move. The town hall is about three
# minutes; the video tokens Gemini charged for it were measured in 9.1.
USD_INR = 85
PRICES = {
    "image (gemini-3.1-flash-image)":      ("per image",        0.039),
    "video via gemini-3.6-flash, LOW":     ("per minute (approx)", 3 * 258 * 1.50 / 1_000_000 * 60 / 60),   # ~1 frame/s x 258 tokens
    "Chirp 3 (9.3, regional)":             ("per audio-minute", 0.016),
}
print(f"{'item':38} {'unit':22} {'USD':>9} {'INR':>8}")
print("-" * 80)
for name, (unit, usd) in PRICES.items():
    print(f"{name:38} {unit:22} {usd:>9.4f} {usd * USD_INR:>8.2f}")
tenants, images, minutes = 5, 200, 120
monthly = tenants * (images * 0.039 + minutes * 0.016)
print(f"\n5 tenants x ({images} images + {minutes} audio minutes) = ${monthly:.2f} = Rs {monthly * USD_INR:,.0f} / month")
print("and DEMO_MODE turns a re-run prompt into Rs 0: the cache key is the prompt's hash.")


## Where this goes
- The UI's **Studio** tab (12.4's `studio.py`) is this notebook's Cell 3 as a page: generate, then read aloud.
- **9.6** is the contract the probe image came back through - `kind`, `media_url`, and the figure shown inline.

## ✅ Lesson 9.4 complete
- ✅ Two buckets read as deployed, nothing created
- ✅ An image through the route; the audit row and the usage row read back
- ✅ Caption, alt text and table as JSON, beside the worker's caption
- ✅ The town hall segmented with the worker's prompt, beside the corpus's segments; the coverage check exercised
- ✅ Diarisation counted against the committed script
- ✅ The 32 MB wall, and a signed PUT that the worker indexed and the lane cited
- ✅ The audit registry refusing an unknown name; `media.py` read from the clone
